# Build email extractor

In [1]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constants import MODEL_LARGE
from dotenv import load_dotenv 

load_dotenv()

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="Summarize the email in 3-4 sentences.")

email_extractor_agent = Agent(MODEL_LARGE, system_prompt="""
    You are customer support agent, your task is to 
    extract relevant information from an email.
""", output_type=EmailExtractor)



TODO: load in the data and test the agent on one email

In [2]:
import pandas as pd 

df = pd.read_json("emails_cleaned.json")
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [3]:
sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [4]:
result = await email_extractor_agent.run(sample_mail)
result

AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been locked out of his account for three days, receiving 'Invalid credentials' errors and failing to receive password reset emails despite multiple attempts across devices and browsers. He has checked spam/junk folders, cleared cache, and tried different browsers without success. He urgently needs access to important work documents before a Friday deadline and requests immediate help to verify his identity and restore account access."))

In [5]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been locked out of his account for three days, receiving 'Invalid credentials' errors and failing to receive password reset emails despite multiple attempts across devices and browsers. He has checked spam/junk folders, cleared cache, and tried different browsers without success. He urgently needs access to important work documents before a Friday deadline and requests immediate help to verify his identity and restore account access.")

In [6]:
isinstance(result.output, BaseModel)

True

In [7]:
result.output.summary

"Oscar Johansson has been locked out of his account for three days, receiving 'Invalid credentials' errors and failing to receive password reset emails despite multiple attempts across devices and browsers. He has checked spam/junk folders, cleared cache, and tried different browsers without success. He urgently needs access to important work documents before a Friday deadline and requests immediate help to verify his identity and restore account access."

TODO: show ground truth and compare

## Load in prompts from mlflow

In [8]:
from mlflow.genai import load_prompt

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4)
    )


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

c:\Users\tasmi\MLOPS\llmops_sadia_awan_mlo25\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson has been locked out of his account for three days due to invalid credentials. He has attempted to reset his password multiple times but the reset emails are not arriving in his inbox, spam, or junk folders. This issue is preventing him from accessing important work documents with a deadline this Friday. He has tried various troubleshooting steps including different browsers and devices without success.')

## LLM Judge

1. require data with columns: inputs, expectations, outputs
2. mlflow experients

In [10]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson has been locked out of his account for three days due to invalid credentials. He has attempted to reset his password multiple times but the reset emails are not arriving in his inbox, spam, or junk folders. This issue is preventing him from accessing important work documents with a deadline this Friday. He has tried various troubleshooting steps including different browsers and devices without success.'}

TODO: dataframe with inputs, expecatations, outputs but only 1 row

In [11]:
df["outputs"] = [{},{}, result.output.model_dump(),{}]

df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...",{}


In [12]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


## LLM Judge

In [13]:
from mlflow.genai.scorers import get_all_scorers

get_all_scorers()

[RetrievalRelevance(name='retrieval_relevance', aggregations=None, description='Evaluate whether each retrieved context chunk is relevant to the input request.', required_columns={'inputs', 'trace'}, inference_params=None, model=None, sample_rate=None, filter_string=None),
 RetrievalSufficiency(name='retrieval_sufficiency', aggregations=None, description='Evaluate whether the information in the last retrieval is sufficient to generate the facts in expected_response or expected_facts.', required_columns={'inputs', 'trace'}, inference_params=None, model=None, sample_rate=None, filter_string=None),
 RetrievalGroundedness(name='retrieval_groundedness', aggregations=None, description='Assess whether the facts in the response are implied by the information in the last retrieval step, i.e., hallucinations do not occur.', required_columns={'inputs', 'trace'}, inference_params=None, model=None, sample_rate=None, filter_string=None),
 ToolCallEfficiency(name='tool_call_efficiency', aggregations=

In [14]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge ="openrouter:/nvidia/nemotron-3-nano-30b-a3b:free" 

with mlflow.start_run(run_name="email-extractor-evalutation"):
    mlflow.log_param("model_judge", llm_judge)
    mlflow.log_param("model", MODEL_LARGE)

    results = mlflow.genai.evaluate(
        data = df_sample,
        scorers=[
            Correctness(model=llm_judge),
            Summarization(model=llm_judge)
        ]
    )

results

2026/04/15 15:17:43 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating: 100%|██████████| 1/1 [Elapsed: 00:06, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: email-extractor-evalutation
  Run ID: acd344fade144784b51ea021a2b769ed

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: acd344fade144784b51ea021a2b769ed
  metrics:
    correctness/mean: 1.0
    summarization/mean: 1.0
  result_df: 1 rows x 15 cols
)

In [ ]:
results.result_df